In [1]:
import subprocess
subprocess.run(['pip', 'install', 'albumentations', '-q'])

import torch
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
print(f"✅ CUDA: {torch.cuda.is_available()}")

✅ GPU: Tesla T4
✅ CUDA: True


In [2]:
from google.colab import files

In [5]:
import os
# نشوف الداتا الموجودة
for item in os.listdir('/kaggle/input'):
    print(item)

datasets


In [7]:
import os

# نشوف كل الـ datasets المتاحة
print("=== كل الداتا ===")
for dataset in os.listdir('/kaggle/input'):
    print(f"\n📁 {dataset}:")
    try:
        for item in os.listdir(f'/kaggle/input/{dataset}'):
            print(f"  • {item}")
    except:
        pass

=== كل الداتا ===

📁 datasets:
  • smadive
  • mohamednasser0123


In [8]:
import os

# الـ paths الصح
print("=== smadive (Pet Disease images) ===")
for item in os.listdir('/kaggle/input/datasets/smadive'):
    print(f"  • {item}")

print("\n=== mohamednasser0123 (pet-dis) ===")
for item in os.listdir('/kaggle/input/datasets/mohamednasser0123'):
    print(f"  • {item}")

=== smadive (Pet Disease images) ===
  • pet-disease-images

=== mohamednasser0123 (pet-dis) ===
  • pet-dis


In [10]:
import os, shutil, cv2
import albumentations as A
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split

# ═══ 1. دمج الداتا ═══
MERGED = '/kaggle/working/merged'
FINAL  = '/kaggle/working/final'

for d in [MERGED, FINAL]:
    if os.path.exists(d): shutil.rmtree(d)
    os.makedirs(d)

def copy_imgs(src):
    for root, dirs, files in os.walk(src):
        if files:
            cls = os.path.basename(root)
            if cls in ['train','valid','test','data','']: continue
            dst = f'{MERGED}/{cls}'
            os.makedirs(dst, exist_ok=True)
            for f in files:
                if f.lower().endswith(('.jpg','.jpeg','.png','.webp')):
                    shutil.copy2(f'{root}/{f}', f'{dst}/{cls}_{len(os.listdir(dst))}_{f}')

copy_imgs('/kaggle/input/datasets/smadive/pet-disease-images')
copy_imgs('/kaggle/input/datasets/mohamednasser0123/pet-dis')
print("✅ الداتا اتدمجت!")

# ═══ 2. شيل الفئات أقل من 80 صورة ═══
for cls in os.listdir(MERGED):
    count = len(os.listdir(f'{MERGED}/{cls}'))
    if count >= 80:
        shutil.copytree(f'{MERGED}/{cls}', f'{FINAL}/{cls}')
    else:
        print(f"❌ شيل: {cls} ({count})")

print(f"\n✅ فئات: {len(os.listdir(FINAL))}")
total = sum(len(os.listdir(f'{FINAL}/{c}')) for c in os.listdir(FINAL))
print(f"✅ صور: {total}")

# ═══ 3. Synthetic Data 750 صورة ═══
SYNTH = '/kaggle/working/synth'
if os.path.exists(SYNTH): shutil.rmtree(SYNTH)
os.makedirs(SYNTH)

transform = A.Compose([
    A.RandomRotate90(p=0.5), A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3), A.ElasticTransform(p=0.3),
    A.GridDistortion(p=0.3), A.GaussNoise(p=0.3),
    A.GaussianBlur(p=0.2), A.CLAHE(p=0.3),
    A.HueSaturationValue(p=0.4),
    A.RandomBrightnessContrast(p=0.4),
])

TARGET = 750
for cls in sorted(os.listdir(FINAL)):
    src = f'{FINAL}/{cls}'
    dst = f'{SYNTH}/{cls}'
    os.makedirs(dst, exist_ok=True)
    imgs = [f for f in os.listdir(src) if f.lower().endswith(('.jpg','.jpeg','.png','.webp'))]
    for f in imgs: shutil.copy2(f'{src}/{f}', f'{dst}/{f}')
    needed = max(0, TARGET - len(imgs))
    for i in range(needed):
        img = cv2.imread(f'{src}/{imgs[i % len(imgs)]}')
        if img is None: continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        aug = transform(image=img)['image']
        cv2.imwrite(f'{dst}/s_{i}_{imgs[i%len(imgs)]}', cv2.cvtColor(aug, cv2.COLOR_RGB2BGR))
    print(f"✅ {cls}: {len(imgs)} → {len(os.listdir(dst))}")

total_synth = sum(len(os.listdir(f'{SYNTH}/{c}')) for c in os.listdir(SYNTH))
print(f"\n🎉 إجمالي: {total_synth} صورة")

# ═══ 4. التدريب ═══
device = torch.device('cuda')

train_tf = transforms.Compose([
    transforms.Resize((260,260)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(20),
    transforms.ColorJitter(0.3,0.3,0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

dataset = datasets.ImageFolder(SYNTH, transform=train_tf)
classes = dataset.classes
num_classes = len(classes)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_data, val_data = random_split(dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_data, batch_size=32, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_data,   batch_size=32, num_workers=2)

print(f"\n✅ فئات: {num_classes} | Train: {train_size} | Val: {val_size}")

model = models.efficientnet_b4(pretrained=True)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0003, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

EPOCHS = 30
best_acc = 0

for epoch in range(EPOCHS):
    model.train()
    correct, total = 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        _, pred = out.max(1)
        total += labels.size(0)
        correct += pred.eq(labels).sum().item()
    train_acc = 100.*correct/total

    model.eval()
    vc, vt = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            _, pred = model(imgs).max(1)
            vt += labels.size(0)
            vc += pred.eq(labels).sum().item()
    val_acc = 100.*vc/vt
    scheduler.step()

    print(f"Epoch {epoch+1}/{EPOCHS} | Train: {train_acc:.1f}% | Val: {val_acc:.1f}%")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save({'model': model.state_dict(), 'classes': classes,
                    'num_classes': num_classes}, '/kaggle/working/best_model_v3.pth')
        print(f"  💾 حُفظ! {best_acc:.1f}%")

print(f"\n🎉 خلص! أحسن دقة: {best_acc:.1f}%")

✅ الداتا اتدمجت!
❌ شيل: Tick Infestation in Dog (77)
❌ شيل: Eye Infection in Dog (69)
❌ شيل: Mange in Dog (71)
❌ شيل: Ringworm in Cat (67)
❌ شيل: Cat Skin Disease.folder (0)
❌ شيل: Scabies in Cat (77)
❌ شيل: Urinary Tract Infection in Cat (70)
❌ شيل: Feline Leukemia (60)
❌ شيل: Feline Panleukopenia (57)
❌ شيل: Distemper in Dog (58)
❌ شيل: Ear Mites in Cat (65)

✅ فئات: 22
✅ صور: 6360
✅ Dental Disease in Cat: 80 → 750
✅ Dental Disease in Dog: 89 → 750
✅ Dermatitis: 787 → 787


libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate


✅ Eye Infection in Cat: 84 → 750
✅ Flea Allergy: 241 → 750
✅ Fungal Infection in Cat: 82 → 750


libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50


✅ Fungal Infection in Dog: 83 → 750
✅ Fungal_infections: 526 → 750
✅ Health: 245 → 750
✅ Healthy: 700 → 750
✅ Hot Spots in Dog: 91 → 750
✅ Hypersensitivity: 322 → 750
✅ Kennel Cough in Dog: 82 → 750
✅ Parvovirus in Dog: 82 → 750
✅ Ringworm: 247 → 750
✅ Scabies: 250 → 750
✅ Skin Allergy in Cat: 93 → 750
✅ Skin Allergy in Dog: 103 → 750
✅ Worm Infection in Cat: 97 → 750
✅ Worm Infection in Dog: 96 → 750
✅ demodicosis: 862 → 862
✅ ringworm: 1118 → 1118

🎉 إجمالي: 17017 صورة

✅ فئات: 22 | Train: 13613 | Val: 3404


...

Epoch 28/30 | Train: 99.4% | Val: 97.2%


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch 29/30 | Train: 99.4% | Val: 97.1%


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch 30/30 | Train: 99.4% | Val: 97.2%

🎉 خلص! أحسن دقة: 97.4%


In [11]:
import shutil
shutil.copy('/kaggle/working/best_model_v3.pth', 
            '/kaggle/working/best_model_FINAL_97.pth')
print("✅ محفوظ!")

# حجم الموديل
import os
size = os.path.getsize('/kaggle/working/best_model_FINAL_97.pth')
print(f"📦 حجم الموديل: {size/1024/1024:.1f} MB")

✅ محفوظ!
📦 حجم الموديل: 67.8 MB
